# UCI TFM and baseline workflow

Primary task: **Dropout=1 vs Graduate=0**; Enrolled is excluded. The split is fixed at 70/15/15 with seed 42. Model selection uses validation data only; the final test split remains untouched until the TFM and explanation protocol are frozen. Second-semester variables are prohibited from the primary feature window.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from src.modeling import (
    SEED, add_first_semester_features, feature_columns, load_primary_data,
    make_splits, run_baseline_validation, run_tfm_validation, save_splits
)

## Data contract and fixed split
The assertions below make target leakage and row overlap fail visibly.

In [2]:
data = add_first_semester_features(load_primary_data())
features = feature_columns(data, 'first_semester')
splits = make_splits(data, seed=SEED)
save_splits(splits)

assert set(data['target'].unique()) == {0, 1}
assert not any(c.startswith('curricular_units_2nd_sem') for c in features)
assert set(splits.train_ids).isdisjoint(splits.validation_ids)
assert set(splits.train_ids).isdisjoint(splits.test_ids)
assert set(splits.validation_ids).isdisjoint(splits.test_ids)

{
    'rows_after_target_filter': len(data),
    'features_before_encoding': len(features),
    'train': len(splits.train_ids),
    'validation': len(splits.validation_ids),
    'test_untouched': len(splits.test_ids),
}

{'rows_after_target_filter': 3630,
 'features_before_encoding': 33,
 'train': 2541,
 'validation': 544,
 'test_untouched': 545}

## Conventional baselines — validation only
All preprocessing is fitted inside each scikit-learn pipeline using training rows only. Results are written to `tables/baseline_validation_metrics.csv`.

In [3]:
baseline_validation = run_baseline_validation()
baseline_validation

,model,split,balanced_accuracy,macro_f1,roc_auc,dropout_recall,confusion_matrix,fit_seconds
0,logistic_regression,validation,0.882239,0.879347,0.945066,0.873239,"[[295, 36], [27, 186]]",0.046211
1,decision_tree_depth_3,validation,0.884586,0.881353,0.923117,0.877934,"[[295, 36], [26, 187]]",0.022301
2,xgboost,validation,0.893997,0.896958,0.945165,0.854460,"[[309, 22], [31, 182]]",0.254401


## TFM feasibility gate
TabICL 2.2.0 is selected from the validation-only feasibility gate: it executed successfully and repeated predictions were identical. TabPFN was not scored because its local weights require separate Prior Labs account/licence acceptance. No TabPFN performance claim is made. The final test set remains untouched until the model and explanation protocol are frozen. Reproduce current validation outputs with `scripts/run_validation.py`.

In [4]:
tfm_validation = run_tfm_validation(candidates=('tabicl',), n_estimators=1)
tfm_validation

,model,status,split,n_estimators,balanced_accuracy,macro_f1,roc_auc,dropout_recall,confusion_matrix,fit_seconds,repeat_prediction_max_abs_diff,model_agnostic_probability_explainer,note
0,tabicl,ok,validation,1,0.892324,0.896576,0.948981,0.84507,"[[311, 20], [33, 180]]",0.382178,0.0,True,
